In [2]:
import pandas as pd
import numpy as np

# --- 1. Configuration ---
# Using the data.csv file as requested.
input_file = '../../data/data_with_features.csv'

# New output file name to reflect the grouping method.
output_file = '../../data/syntheticData/synthetic_data_bootstrap_by_treatment_id.csv'

# The block size for resampling.
block_size = 15

# --- 2. Load the Original Data ---
try:
    df = pd.read_csv(input_file)
    print(f"✅ Successfully loaded original data from '{input_file}'.")
    print(f"   Original dataset shape: {df.shape}")
except FileNotFoundError:
    print(f"❌ Error: The file '{input_file}' was not found. Please make sure it's uploaded or the path is correct.")
    # Stop execution if the file isn't found
    exit()

# --- 3. Perform Grouped Block Bootstrapping by 'treatment_id' ---
print("\n⚙️  Starting GROUPED block bootstrapping process using 'treatment_id'...")

all_synthetic_groups = []
# Group the DataFrame by the 'treatment_id' column.
grouped_data = df.groupby('treatment_id')
num_groups = len(grouped_data)

print(f"   Found {num_groups} unique time series based on 'treatment_id'.")

# Process each group (each independent time series) one by one
for group_name, group_df in grouped_data:
    n_rows_group = len(group_df)
    
    # This logic correctly handles variable lengths.
    # If a group is too short for a full block, we simply resample its rows.
    if n_rows_group < block_size:
        synthetic_group = group_df.sample(n=n_rows_group, replace=True)
    else:
        synthetic_blocks = []
        num_blocks_to_generate = int(np.ceil(n_rows_group / block_size))

        for _ in range(num_blocks_to_generate):
            start_index = np.random.randint(0, n_rows_group - block_size + 1)
            block = group_df.iloc[start_index : start_index + block_size]
            synthetic_blocks.append(block)

        # Combine blocks and trim to the original group's length
        synthetic_group = pd.concat(synthetic_blocks, ignore_index=True)
        synthetic_group = synthetic_group.iloc[:n_rows_group]

    all_synthetic_groups.append(synthetic_group)

# Combine all the processed synthetic groups back into one DataFrame
final_synthetic_df = pd.concat(all_synthetic_groups, ignore_index=True)

# --- 4. Save the Synthetic Data ---
final_synthetic_df.to_csv(output_file, index=False)

print("\n✅ Process complete!")
print(f"   Synthetic data correctly generated and saved to '{output_file}'.")
print(f"   Final dataset shape: {final_synthetic_df.shape}")

# Optional: Display the first few rows of the new synthetic dataset
print("\n--- Head of the new synthetic dataset ---")
print(final_synthetic_df.head())

✅ Successfully loaded original data from '../../data/data_with_features.csv'.
   Original dataset shape: (3786, 14)

⚙️  Starting GROUPED block bootstrapping process using 'treatment_id'...
   Found 224 unique time series based on 'treatment_id'.

✅ Process complete!
   Synthetic data correctly generated and saved to '../../data/syntheticData/synthetic_data_bootstrap_by_treatment_id.csv'.
   Final dataset shape: (3786, 14)

--- Head of the new synthetic dataset ---
  treatment_id date_of_introduction observation_date  initial_female_count  \
0         T1R1           2019-07-31       2019-08-02                     1   
1         T1R1           2019-07-31       2019-08-03                     1   
2         T1R1           2019-07-31       2019-08-04                     1   
3         T1R1           2019-07-31       2019-08-05                     1   
4         T1R1           2019-07-31       2019-08-06                     1   

   leaf_area_cm2  population_density  replicate_id  cumulativ